# Exploratory Data Analysis and Cleaning

### Inputs: 
- `"../data/admissions/Admissions Data_6.26.25.xlsx`
- `"../data/career-services/Admissions Data.xlsx"`
- `"../data/career-services/Interviews.xlsx"`
- `"../data/career-services/Outcome Data.xlsx"`

### Outputs:
- `"../data/derived/admissions.csv"`
- `"../data/derived/interviews.csv"`
- `"../data/derived/outcomes.csv"`
- `"../data/derived/companies.txt"`
  

### Purpose:

1. Clean and merge the original datasets.
2. Get list of all company names mentioned in the records.

We received two sets of data: 

- Data from Admissions
- Data from Career Services

It appears that there is some overlap in the provided data. Let's take a closer look at that.

In [1]:
import pandas as pd

admissions_data = pd.read_excel("../data/admissions/Admissions Data_6.26.25.xlsx")
admissions_data = admissions_data.rename(columns={"StudentID": "Student ID"})
cs_admissions_data = pd.read_excel("../data/career-services/Admissions Data.xlsx")
interviews = pd.read_excel("../data/career-services/Interviews.xlsx")
outcomes = pd.read_excel("../data/career-services/Outcome Data.xlsx")

In [2]:
print(f"{admissions_data["Student ID"].nunique() = }")
print(f"{cs_admissions_data["Student ID"].nunique() = }")
print(f"{interviews["Student ID"].nunique() = }")
print(f"{outcomes["Student ID"].nunique() = }")

admissions_data["Student ID"].nunique() = 2851
cs_admissions_data["Student ID"].nunique() = 2433
interviews["Student ID"].nunique() = 1749
outcomes["Student ID"].nunique() = 2287


In [3]:
admissions_data.head()

,Student ID,Application Slate ID,Application Entry Term,Decision Released Name,Waitlisted,Birthdate,Pronouns,Application Programs,Marital or Partnership Status,Do you have children?,...,School - Degree Conferred,School - Institution.1,School - City.1,School - Region.1,School - Country.1,School - GPA.1,School - GPA Scale.1,School - Major 1.1,School - Degree.1,School - Degree Conferred.1
0,2.400440e+14,396167970,August 2017,Enrolling (Dep. Rec'd),No,1990-05-30,NaN,MBA,Will marry before matriculation,0.0,...,2012-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
1,5.400161e+14,974477819,August 2025,Enrolling (Dep. Rec'd),No,2000-06-12,She/Her/Hers,MBA,Single,0.0,...,2022-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
2,5.400161e+14,838718625,August 2025,Enrolling (Dep. Rec'd),No,1997-05-30,She/Her/Hers,MBA,Single,0.0,...,NaT,Monterey Inst Intrntl Studies,Monterey,CA,United States,3.9,4.0,International Relations,NaN,NaT
3,5.400161e+14,954559889,August 2024,Enrolling (Dep. Rec'd),No,1997-06-27,She/Her/Hers,MBA,Single,0.0,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
4,5.000161e+14,578268730,August 2018,Enrolling (Dep. Rec'd),No,1982-08-17,NaN,MBA,Single,0.0,...,2004-05-01,University Of Oxford,Oxford Uk OX1 2,England,United Kingdom,0.0,0.0,English Language and Literature,Doctor of Philosophy,2015-11-01


In [ ]:
cs_admissions_data.head()

The admissions data from Admissions and Career Services share the Student ID. Let's merge them by adding whatever information we don't already have in `admissions_data`:

In [ ]:
# Step 1: Perform a left join to keep all entries from admissions_data
merged_df = admissions_data.dropna(subset="Student ID").merge(
    cs_admissions_data,
    on="Student ID",
    how="left",
    suffixes=("", "_cs"),
)

# Step 2: Get list of overlapping columns (excluding the join key)
overlapping_columns = [
    col
    for col in admissions_data.columns
    if col in cs_admissions_data.columns and col != "Student ID"
]

# Step 3: Fill missing values in overlapping columns
for col in overlapping_columns:
    merged_df[col] = merged_df[col].fillna(merged_df[f"{col}_cs"])

# Step 4: Add new columns from cs_admissions_data that don't exist in admissions_data
cs_only_columns = [
    col
    for col in cs_admissions_data.columns
    if col not in admissions_data.columns and col != "Student ID"
]

for col in cs_only_columns:
    merged_df[col] = merged_df[f"{col}"]

# Step 5: Drop the duplicate '_cs' columns
columns_to_drop = [col for col in merged_df.columns if col.endswith("_cs")]
admissions_data = merged_df.drop(columns=columns_to_drop)

We can only work with the students that we have outcome data and admissions data for. Missing interview data is acceptable.

In [ ]:
print(f"{admissions_data["Student ID"].nunique() = }")

In [ ]:
student_ids = {
    *admissions_data["Student ID"].dropna().astype(int).tolist()
}.intersection(
    {*outcomes["Student ID"].dropna().astype(int).tolist()},
)
len(student_ids)

In [ ]:
admissions_data.loc[
    ~admissions_data["Student ID"].isin(student_ids), "Grad Year"
].value_counts()

Prepare a list of students that appear in `admissions` but not in `outcomes`:

In [ ]:
admissions_data[~admissions_data["Student ID"].isin(outcomes["Student ID"])].to_csv(
    "../data/derived/students_missing_from_outcomes.csv",
    index=False,
)

Prepare a list of students that appear in `outcomes` but not in `admissions`:

In [ ]:
outcomes[~outcomes["Student ID"].isin(admissions_data["Student ID"])].to_csv(
    "../data/derived/students_missing_from_admissions.csv",
    index=False,
)

Looks like we have data for `1928` different students. The dropped students are of the 2025 to 2027 grad year, which we don't have outcome data for, yet. Let's drop all records from other students:

In [ ]:
admissions_data = admissions_data[admissions_data["Student ID"].isin(student_ids)]
interviews = interviews[interviews["Student ID"].isin(student_ids)]
outcomes = outcomes[outcomes["Student ID"].isin(student_ids)]

The data types of the columns are not necessarily conducisive to further analysis, let's fix that:

In [ ]:
admissions_data["Student ID"] = admissions_data["Student ID"].astype(int)

Let's take a look at missing values:

In [ ]:
admissions_data.isna().sum()

There are quite a few missing values. We will need to discuss how to treat them: Drop or impute?

Suggestion:
- `Marital or Partnership Status`: Missing value becomes all `False` in dummy encoding
- `Do you have children?`: Assume `NA` means `0`
- `Race`: Missing value becomes all `False` in dummy encoding
- `Are you being sponsored?`: Assume `NA` means `0`
- `Job 1 Bonus`: Assume `NA` means `0`
- Rest of `Job 1` columns: Leaning towards dropping missing values, but we should discuss

## Prepare list of companies

To prepare the gender-coding, we need a list of all companies mentioned in the job history, interviews, and outcomes.

In [ ]:
companies = set(
    admissions_data["Job 1 Organization"].unique().tolist()
    + interviews["Employer"].unique().tolist()
    + outcomes["Employer"].unique().tolist()
)

In [ ]:
with open("../data/derived/companies.txt", "w+") as f:
    f.writelines("\n".join(map(str, companies)))

In [ ]:
admissions_data.to_csv("../data/derived/admissions.csv", index=False)
interviews.to_csv("../data/derived/interviews.csv", index=False)
outcomes.to_csv("../data/derived/outcomes.csv", index=False)